# 04 — Kết quả binary và test chéo

Pooled và từng client: Accuracy, Precision/Recall/F1 Attack, FAR, ROC-AUC, PR-AUC, confusion matrix Benign/Attack và tỷ lệ phát hiện theo Attack gốc. Chỉ số CSV ở thang 0–1. Local chéo và FL phải so trên cùng tập test của client đích.

In [ ]:
from pathlib import Path
import os, sys, json, importlib
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p/'COPY_PROVENANCE.json').is_file()
               and (p/'configs/tuning.json').is_file() and (p/'notebook_workflow.py').is_file())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for module_name in ('common', 'feature_preprocessing', 'prepare', 'models', 'metrics', 'notebook_workflow'):
    module = importlib.import_module(module_name)
    assert Path(module.__file__).resolve().parent == PROJECT.resolve(), 'Restart kernel to load this copy.'
    importlib.reload(module)
workflow = sys.modules['notebook_workflow']
CONFIG = PROJECT/'configs/tuning.json'
C = workflow.settings(CONFIG)
RUN_ROOT = workflow.run_root(C)
print('Project:', PROJECT)
print('Results:', RUN_ROOT)
display(workflow.budget(CONFIG))

In [ ]:
import pandas as pd
frame = workflow.report(CONFIG)
if not frame.empty:
    frame = frame[frame["scenario"] != "fedavg_uniform"].copy()
if frame.empty:
    print("Chưa có kết quả test. Chạy 02 rồi 03.")
else:
    display(frame)
    display(frame[frame["test_client"] == "pooled"].pivot_table(index="model",columns="scenario",values=["accuracy","f1","recall","far"]))

In [ ]:
if not frame.empty:
    comparisons = []
    for model in C["models"]:
        for client in ("cic_1","cic_2","cic_3"):
            for target in ("cic_1","cic_2","cic_3"):
                if target == client: continue
                local = frame[(frame["model"]==model)&(frame["scenario"]=="local_"+client)&(frame["test_client"]==target)]
                for scenario in ("fedavg_weighted","fedadam_weighted"):
                    fl = frame[(frame["model"]==model)&(frame["scenario"]==scenario)&(frame["test_client"]==target)]
                    if local.empty or fl.empty: continue
                    a,b = local.iloc[0],fl.iloc[0]
                    comparisons.append({"model":model,"local_train_client":client,"test_client":target,"fl_scenario":scenario,"local_accuracy":a.accuracy,"fl_accuracy":b.accuracy,"local_f1":a.f1,"fl_f1":b.f1,"f1_gain":b.f1-a.f1})
    cross = pd.DataFrame(comparisons)
    display(cross)
    if not cross.empty: display(cross[cross["local_train_client"]=="cic_1"])

In [ ]:
if not frame.empty:
    attacks = pd.read_csv(RUN_ROOT/"results/attack_detection_rates.csv")
    per_class = pd.read_csv(RUN_ROOT/"results/per_class_metrics.csv")
    display(attacks[attacks["scenario"] != "fedavg_uniform"])
    display(per_class[per_class["scenario"] != "fedavg_uniform"])
    matrices = json.loads((RUN_ROOT/"results/confusion_matrices.json").read_text())
    display([m for m in matrices if m["scenario"] != "fedavg_uniform"])